In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.metrics import confusion_matrix, roc_curve, auc, f1_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import *
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

# ==========================================
# 1. MOUNT DRIVE & SETUP PATHS
# ==========================================
drive.mount('/content/drive')

# UPDATE THIS: Path to your 'skin cancer dataset' folder in Drive
BASE_PATH = '/content/drive/MyDrive/skin_cancer_dataset'
TRAIN_DIR = os.path.join(BASE_PATH, 'train')
TEST_DIR = os.path.join(BASE_PATH, 'test')

# Path to save weights so you can resume if Colab restarts
CHECKPOINT_PATH = os.path.join(BASE_PATH, 'vital_nexus_checkpoint.weights.h5')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16  # Reduced to 16 to prevent RAM crashes on Dual-Path models
EPOCHS = 20      # Total target epochs

# ==========================================
# 2. DATA AUGMENTATION (For small 3033 image set)
# ==========================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary')
test_gen = test_datagen.flow_from_directory(TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

# ==========================================
# 3. DATASET WRAPPER (Handles Dual-Input & Signature)
# ==========================================
def make_dataset(generator):
    def gen_func():
        for x, y in generator:
            yield (x, x), y

    sig = ((tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32)),
           tf.TensorSpec(shape=(None,), dtype=tf.float32))
    return tf.data.Dataset.from_generator(gen_func, output_signature=sig)

# ==========================================
# 4. NOVEL ARCHITECTURE: ATTENTION-FUSION
# ==========================================
def build_novel_model():
    # Input for Dual-Path (Path 1: Textures, Path 2: Structure)
    in1, in2 = Input(shape=(224,224,3)), Input(shape=(224,224,3))

    # Path 1: EfficientNetB0 (Deep Features)
    base_1 = EfficientNetB0(weights='imagenet', include_top=False)(in1)
    x1 = GlobalAveragePooling2D()(base_1)

    # Path 2: MobileNetV2 (Spatial Features)
    base_2 = MobileNetV2(weights='imagenet', include_top=False)(in2)
    x2 = GlobalAveragePooling2D()(base_2)

    # Feature Fusion & Channel Attention
    merged = Concatenate()([x1, x2])
    attn = Dense(merged.shape[-1], activation='softmax')(merged)
    refined = Multiply()([merged, attn])

    # Triage Decision Head
    head = Dense(512, activation='relu')(refined)
    out = Dense(1, activation='sigmoid')(Dropout(0.5)(head))

    model = Model(inputs=[in1, in2], outputs=out)
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# ==========================================
# 5. TRAINING WITH AUTO-RESUME
# ==========================================
train_ds = make_dataset(train_gen)
test_ds = make_dataset(test_gen)

model = build_novel_model()

# Save checkpoint after every epoch if accuracy improves
ckpt = ModelCheckpoint(filepath=CHECKPOINT_PATH, save_weights_only=True,
                       monitor='accuracy', mode='max', save_best_only=True, verbose=1)

# Check if we should resume
if os.path.exists(CHECKPOINT_PATH):
    print(f"\n[RESUME] Loading weights from {CHECKPOINT_PATH}...")
    model.load_weights(CHECKPOINT_PATH)
else:
    print("\n[START] No checkpoint found. Starting fresh training...")

# Start Training
history = model.fit(train_ds, steps_per_epoch=len(train_gen), epochs=EPOCHS, callbacks=[ckpt])

# ==========================================
# 6. FINAL EVALUATION & RESULTS
# ==========================================
y_true = test_gen.classes
probs = model.predict(test_ds, steps=len(test_gen)).flatten()
preds = (probs > 0.5).astype(int)

# Metrics for Medical Guidance (Sensitivity/Specificity)
tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
acc, sens, spec = accuracy_score(y_true, preds), tp/(tp+fn), tn/(tn+fp)
f1, fpr, tpr = f1_score(y_true, preds), *roc_curve(y_true, probs)[:2]
roc_auc = auc(fpr, tpr)

# Display Results Table
print("\n" + "="*30 + "\nFINAL PERFORMANCE METRICS\n" + "="*30)
results = pd.DataFrame({"Metric": ["Accuracy", "Sensitivity", "Specificity", "F1-Score", "AUC"],
                        "Value": [f"{acc:.4f}", f"{sens:.4f}", f"{spec:.4f}", f"{f1:.4f}", f"{roc_auc:.4f}"]})
print(results.to_string(index=False))

# Visualization: Confusion Matrix (Clinical Triage)
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_true, preds), annot=True, fmt='d', cmap='Reds',
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
plt.title('Confusion Matrix: Vital Nexus AI'); plt.show()

# Visualization: AUC Curve
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='red', lw=2, label=f'ROC AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('Receiver Operating Characteristic'); plt.legend(); plt.show()

Mounted at /content/drive
Found 1654 images belonging to 2 classes.
Found 660 images belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/tmp/ipykernel_6498/3769984932.py:77: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_2 = MobileNetV2(weights='imagenet', include_top=False)(in2)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

[RESUME] Loading weights from /content/drive/MyDrive/skin_cancer_dataset/vital_nexus_checkpoint.weights.h5...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 748 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.8747 - loss: 0.4410
Epoch 1: accuracy improved from None to 0.87062, saving model to /content/drive/MyDrive/skin_cancer_dataset/vital_nexus_checkpoint.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/skin_cancer_dataset/vital_nexus_checkpoint.weights.h5
104/104 ━━━━━━━━━━━━━━━━━━━━ 1037s 9s/step - accuracy: 0.8706 - loss: 0.3982
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8676 - loss: 0.3104
Epoch 2: accuracy did not improve from 0.87062
104/104 ━━━━━━━━━━━━━━━━━━━━ 854s 8s/step - accuracy: 0.8706 - loss: 0.2979
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8800 - loss: 0.2650
Epoch 3: accuracy did not improve from 0.87062
104/104 ━━━━━━━━━━━━━━━━━━━━ 850s 8s/step - accuracy: 0.8706 - loss: 0.2733
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8714 - loss: 0.2585
Epoch 4: accuracy did not improve from 0.87062
104/104 ━━━━━━━━━━━━━━━━━━━━ 844s 8s/step - acc